# Figure 2D | Cross-platform integration and cell-type annotation

Integration of scRNA-seq, lower-plex BASSFISH (`h_bass`), and higher-plex BASSFISH (`t_bass`) using shared genes and Harmony, followed by hierarchical cell-type refinement, UMAP visualization, marker-gene dot plots, and comparison between technology-specific and integrated annotations.

Check all input/output paths before running. Run the notebook from top to bottom with a fresh kernel.

| Workflow | Purpose |
|---|---|
| 1 | Load and subset the three datasets |
| 2 | Restrict to shared genes and concatenate cells |
| 3 | Normalize, regress total counts, PCA, Harmony, neighbors and UMAP |
| 4 | Broad Leiden clustering and removal of tiny outlier clusters |
| 5 | Hierarchical refinement of major neuronal/vascular populations |
| 6 | Load the final cleaned integrated annotation used for plotting |
| 7 | UMAP and marker-gene dot plot |
| 8 | Annotation agreement between technology-specific and integrated labels |

The source notebooks contained some manual curation steps that were saved into a final cleaned h5ad file. The explicitly recorded subclustering steps are retained below; final manuscript plots use the cleaned integrated object.


## Imports and configuration

The original analysis used absolute server paths. For GitHub release, configure the project-relative paths below to point to the corresponding input files. The analysis parameters are retained from the source notebooks.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb

warnings.filterwarnings("ignore")

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 7,
    "axes.labelsize": 7,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---------------------------------------------------------------------
# Paths: edit these for the local data layout.
# ---------------------------------------------------------------------
DATA_ROOT = Path("data")
OUT_DIR = Path("outputs/Fig2D")
FIG_DIR = OUT_DIR / "figures"
INTERMEDIATE_DIR = OUT_DIR / "integrated_from_counts"

FIG_DIR.mkdir(parents=True, exist_ok=True)
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)

T_BASS_PATH = DATA_ROOT / "1000genes/integrated_major_clustered_refined_domain_temp.h5ad"
H_BASS_ANNO_PATH = DATA_ROOT / "200genes/integrated_refined_cluster_refined_domain_20250605.h5ad"
H_BASS_RAW_PATH = DATA_ROOT / "200genes/integrated_raw.h5ad"
SCRNA_ANNO_PATH = DATA_ROOT / "scRNA_seq/integrated_refined_clusters.h5ad"
SCRNA_RAW_PATH = DATA_ROOT / "scRNA_seq/integrated_raw.h5ad"

# Final manually cleaned annotation generated from the refinement workflow.
FINAL_ANNOTATED_PATH = (
    DATA_ROOT / "integrated_from_counts/"
    "integrated_3_tech_selected_filtered_from_raw_anno_clean.h5ad"
)

TECH_COLORS = {
    "scrna": "#2acaea",
    "h_bass": "#ffa500",
    "t_bass": "#ff4040",
}


## 1. Load and subset the three datasets

The higher-plex and lower-plex BASSFISH datasets are restricted to cortex/striatum-related spatial domains. The scRNA-seq reference excludes annotation classes that were not used in this cortex/striatum integration. Raw-count expression is restored for the lower-plex BASSFISH and scRNA-seq datasets before cross-platform normalization.


In [ ]:
# Higher-plex BASSFISH.
t_adata = sc.read_h5ad(T_BASS_PATH)

t_adata_selected = t_adata[
    t_adata.obs["refined_domain"].isin(["ctx_0", "ctx_1", "ctx_2", "str_0"])
].copy()

print("t_bass selected:", t_adata_selected.shape)


In [ ]:
# Lower-plex BASSFISH annotation object.
h_adata = sc.read_h5ad(H_BASS_ANNO_PATH)
h_adata = h_adata.raw.to_adata()

# Restore the corresponding raw-count matrix used in the source workflow.
h_adata_raw = sc.read_h5ad(H_BASS_RAW_PATH)
h_adata_raw.obs.index = (
    h_adata_raw.obs["batch"].astype(str)
    + "_"
    + h_adata_raw.obs.index.astype(str)
)
h_adata_raw = h_adata_raw[h_adata.obs_names].copy()
h_adata_raw = h_adata_raw.raw.to_adata()

h_adata.X = h_adata_raw.X.copy()

h_adata_selected = h_adata[
    h_adata.obs["cast_domain"].isin(
        ["major_4", "major_9", "major_10", "major_5", "major_11", "major_18"]
    )
].copy()

print("h_bass selected:", h_adata_selected.shape)


In [ ]:
# scRNA-seq reference.
sc_adata = sc.read_h5ad(SCRNA_ANNO_PATH)
sc_adata = sc_adata.raw.to_adata()

sc_adata_raw = sc.read_h5ad(SCRNA_RAW_PATH)
sc_adata_raw = sc_adata_raw[sc_adata.obs_names].copy()
sc_adata.X = sc_adata_raw.X.copy()

EXCLUDED_SCRNA_CLASSES = [
    "In_Olf_1", "In_Olf_2", "Mix", "Mt_neuron",
    "Ex_HP_1", "Ex_HP_2", "Ex_HP_3", "Ex_Unknown",
    "Mix_micro_oligo", "Mix_astro_oligo",
    "In_Unknown_1", "In_Unknown_2",
    "Mix_micro_astro", "Nonneural_unknown",
]

sc_adata_selected = sc_adata[
    ~sc_adata.obs["refined_clusters"].isin(EXCLUDED_SCRNA_CLASSES)
].copy()

print("scrna selected:", sc_adata_selected.shape)


## 2. Shared genes and concatenation

Only genes present in all three selected datasets are retained. Cells are concatenated with a `tech` label (`scrna`, `h_bass`, or `t_bass`). Cross-platform batch and sample identifiers are prefixed by technology before Harmony integration.


In [ ]:
overlap_genes = [
    g for g in t_adata_selected.var_names
    if g in h_adata_selected.var_names and g in sc_adata_selected.var_names
]

print("Shared genes:", len(overlap_genes))

t_adata_selected = t_adata_selected[:, overlap_genes].copy()
h_adata_selected = h_adata_selected[:, overlap_genes].copy()
sc_adata_selected = sc_adata_selected[:, overlap_genes].copy()

integrated_adata = sc.concat(
    [sc_adata_selected, h_adata_selected, t_adata_selected],
    label="tech",
    keys=["scrna", "h_bass", "t_bass"],
    index_unique="_",
    join="outer",
)

print("Integrated object:", integrated_adata.shape)


In [ ]:
# Build region labels used to inspect the integrated space.
integrated_adata.obs["final_domain"] = integrated_adata.obs.apply(
    lambda x: f"{x['tech']}_{x['cast_domain']}", axis=1
)

cortex_set = {
    "t_bass_major_3",
    "h_bass_major_5",
    "h_bass_major_11",
    "h_bass_major_18",
}

striatum_set = {
    "t_bass_major_4",
    "h_bass_major_4",
    "h_bass_major_9",
    "h_bass_major_10",
}

def assign_region(row):
    value = row["final_domain"]

    if value in cortex_set:
        return "Cortex"
    if value in striatum_set:
        return "Striatum"
    if value == "scrna_nan":
        return row["Brain_region"]
    return "Unknown"

integrated_adata.obs["final_region"] = integrated_adata.obs.apply(
    assign_region, axis=1
)

integrated_adata.obs["final_batch"] = integrated_adata.obs.apply(
    lambda x: f"{x['tech']}_{x['batch']}", axis=1
)

integrated_adata.obs["final_sample"] = integrated_adata.obs.apply(
    lambda x: f"{x['tech']}_{x['sample']}", axis=1
)


## 3. QC, normalization and Harmony integration

The source workflow normalized each cell to the Scanpy default total, log-transformed the data, regressed out total counts, scaled values to a maximum absolute value of 10, performed PCA, and integrated the PCA representation with Harmony using technology and technology-specific batch as covariates.

Harmony parameters are retained from the original workflow: `theta=[2.5, 1.5]`, `max_iter_harmony=10`. A 15-nearest-neighbor graph was constructed from the first 30 Harmony-corrected PCs, followed by UMAP with `min_dist=0.1`.


In [ ]:
sc.pp.calculate_qc_metrics(
    integrated_adata,
    percent_top=None,
    log1p=False,
    inplace=True,
)

sc.pl.violin(
    integrated_adata,
    ["n_genes_by_counts", "total_counts"],
    jitter=0,
    stripplot=False,
    multi_panel=True,
    groupby="tech",
    palette=TECH_COLORS,
    show=False,
)
plt.savefig(
    FIG_DIR / "Fig2D_three_technology_QC_violin.pdf",
    dpi=300,
    bbox_inches="tight",
    transparent=True,
)
plt.close()


In [ ]:
def preprocess_and_integrate(adata):
    adata = adata.copy()

    print("Input shape:", adata.shape)

    adata.layers["raw"] = adata.X.copy()

    sc.pp.normalize_total(adata, target_sum=None)
    sc.pp.log1p(adata)
    sc.pp.regress_out(adata, ["total_counts"])
    sc.pp.scale(adata, max_value=10)

    sc.tl.pca(adata, svd_solver="arpack")

    sc.external.pp.harmony_integrate(
        adata,
        key=["tech", "final_batch"],
        basis="X_pca",
        adjusted_basis="X_pca_harmony",
        lamb=None,
        theta=[2.5, 1.5],
        max_iter_harmony=10,
    )

    sc.pp.neighbors(
        adata,
        n_neighbors=15,
        n_pcs=30,
        use_rep="X_pca_harmony",
    )
    sc.tl.umap(adata, min_dist=0.1)

    return adata


integrated_adata = preprocess_and_integrate(integrated_adata)


## 4. Broad Leiden clustering

Broad clusters were generated at Leiden resolution 0.2. In the source analysis, clusters 0–10 were retained and the very small higher-numbered clusters were excluded as outliers before hierarchical refinement.


In [ ]:
sc.tl.leiden(integrated_adata, resolution=0.2)

print(integrated_adata.obs["leiden"].value_counts())

KEEP_BROAD_CLUSTERS = [str(i) for i in range(11)]

integrated_adata_filtered = integrated_adata[
    integrated_adata.obs["leiden"].isin(KEEP_BROAD_CLUSTERS)
].copy()

print("After filtering:", integrated_adata_filtered.shape)


In [ ]:
# Diagnostic UMAPs.
sc.settings.set_figure_params(figsize=(8, 8))

sc.pl.umap(
    integrated_adata_filtered,
    color="tech",
    palette=TECH_COLORS,
    size=1,
    show=False,
)
plt.savefig(FIG_DIR / "Fig2D_UMAP_by_technology_initial.pdf", bbox_inches="tight")
plt.close()

sc.pl.umap(
    integrated_adata_filtered,
    color="leiden",
    legend_loc="on data",
    size=1,
    show=False,
)
plt.savefig(FIG_DIR / "Fig2D_UMAP_broad_Leiden.pdf", bbox_inches="tight")
plt.close()

sc.pl.umap(
    integrated_adata_filtered,
    color="final_region",
    size=1,
    show=False,
)
plt.savefig(FIG_DIR / "Fig2D_UMAP_region_initial.pdf", bbox_inches="tight")
plt.close()


In [ ]:
integrated_adata.write_h5ad(
    INTERMEDIATE_DIR / "integrated_3_tech_selected_from_raw.h5ad"
)
integrated_adata_filtered.write_h5ad(
    INTERMEDIATE_DIR / "integrated_3_tech_selected_filtered_from_raw.h5ad"
)


## 5. Hierarchical refinement of integrated clusters

The source annotation workflow further subdivided selected broad Leiden populations. The hidden cross-notebook dependencies have been made explicit here:

- broad cluster 1 → striatal neurons;
- broad cluster 0 → upper/middle-layer cortical excitatory neurons;
- broad clusters 6 and 8 → additional cortical excitatory neurons;
- broad cluster 3 → inhibitory neurons;
- broad cluster 4 → vascular cells.

The mappings below reproduce the explicitly recorded manual cluster assignments in the source notebook. Additional curation used for the final manuscript annotation was stored in the final cleaned h5ad file loaded in the next section.


In [ ]:
# Initialize the integrated annotation column if needed.
if "refined_combined_cluster" not in integrated_adata_filtered.obs:
    integrated_adata_filtered.obs["refined_combined_cluster"] = "Unassigned"
else:
    integrated_adata_filtered.obs["refined_combined_cluster"] = (
        integrated_adata_filtered.obs["refined_combined_cluster"].astype(str)
    )


### Striatal neurons


In [ ]:
str_neuron = integrated_adata_filtered[
    integrated_adata_filtered.obs["leiden"] == "1"
].copy()

sc.tl.leiden(str_neuron, resolution=0.5)

STRIATAL_CLUSTER_MAP = {
    "0": "MSN_D1",
    "1": "MSN_D2",
    "2": "MSN_Otof",
    "3": "MSN_Rreb1",
    "4": "Mix_str",
    "5": "MSN_D1",
    "6": "Chat",
    "7": "MSN_D2",
    "8": "MSN_D1",
    "9": "Unknown",
    "10": "MSN_D2",
}

str_neuron.obs["refined_combined_cluster"] = (
    str_neuron.obs["leiden"].map(STRIATAL_CLUSTER_MAP).astype(str)
)

integrated_adata_filtered.obs.loc[
    str_neuron.obs_names, "refined_combined_cluster"
] = str_neuron.obs["refined_combined_cluster"]

str_neuron.write(INTERMEDIATE_DIR / "str_neuron.h5ad")


### Cortical excitatory neurons: broad cluster 0


In [ ]:
ctx_neuron = integrated_adata_filtered[
    integrated_adata_filtered.obs["leiden"] == "0"
].copy()

sc.tl.leiden(ctx_neuron, resolution=0.3)

CTX_CLUSTER0_MAP = {
    "0": "Ex_L2/3_1",
    "1": "Ex_L4/5",
    "2": "Ex_L2/3_2",
    "3": "Ex_L4/5",
    "4": "Ex_L5_1",
    "5": "Ex_L4/5",
    "6": "Unknown",
    "7": "Unknown",
    "8": "Unknown",
}

ctx_neuron.obs["refined_combined_cluster"] = (
    ctx_neuron.obs["leiden"].map(CTX_CLUSTER0_MAP).astype(str)
)

integrated_adata_filtered.obs.loc[
    ctx_neuron.obs_names, "refined_combined_cluster"
] = ctx_neuron.obs["refined_combined_cluster"]

ctx_neuron.write(INTERMEDIATE_DIR / "ctx_neuron.h5ad")


### Cortical excitatory neurons: broad clusters 6 and 8


In [ ]:
ctx_neuron2 = integrated_adata_filtered[
    integrated_adata_filtered.obs["leiden"].isin(["6", "8"])
].copy()

sc.tl.leiden(ctx_neuron2, resolution=0.3)

CTX_CLUSTER68_MAP = {
    "0": "Ex_L6",
    "1": "Ex_L5_2",
    "2": "Ex_L7",
    "3": "Ex_later",
    "4": "Ex_L6",
}

ctx_neuron2.obs["refined_combined_cluster"] = (
    ctx_neuron2.obs["leiden"].map(CTX_CLUSTER68_MAP).astype(str)
)

integrated_adata_filtered.obs.loc[
    ctx_neuron2.obs_names, "refined_combined_cluster"
] = ctx_neuron2.obs["refined_combined_cluster"]

ctx_neuron2.write(INTERMEDIATE_DIR / "ctx_neuron2.h5ad")


### Inhibitory neurons


In [ ]:
in_neuron = integrated_adata_filtered[
    integrated_adata_filtered.obs["leiden"] == "3"
].copy()

sc.tl.leiden(in_neuron, resolution=0.2)

INHIBITORY_CLUSTER_MAP = {
    "0": "In_Pvalb_1",
    "1": "In_Sst_1",
    "2": "In_Pvalb_2",
    "3": "In_Lamp5",
    "4": "In_Sst_2",
}

in_neuron.obs["refined_combined_cluster"] = (
    in_neuron.obs["leiden"].map(INHIBITORY_CLUSTER_MAP).astype(str)
)

integrated_adata_filtered.obs.loc[
    in_neuron.obs_names, "refined_combined_cluster"
] = in_neuron.obs["refined_combined_cluster"]

in_neuron.write(INTERMEDIATE_DIR / "in_neuron.h5ad")


### Vascular cells


In [ ]:
vascular = integrated_adata_filtered[
    integrated_adata_filtered.obs["leiden"] == "4"
].copy()

sc.tl.leiden(vascular, resolution=0.2)

VASCULAR_CLUSTER_MAP = {
    "0": "Endo",
    "1": "Peri",
    "2": "Unknown",
}

vascular.obs["refined_combined_cluster"] = (
    vascular.obs["leiden"].map(VASCULAR_CLUSTER_MAP).astype(str)
)

integrated_adata_filtered.obs.loc[
    vascular.obs_names, "refined_combined_cluster"
] = vascular.obs["refined_combined_cluster"]

vascular.write(INTERMEDIATE_DIR / "vascular.h5ad")


## 6. Final cleaned integrated annotation

The original second notebook loaded a manually cleaned integrated object for the manuscript UMAP, marker dot plot, and annotation-comparison panels. This file includes curation beyond the explicit refinement mappings recorded above and is therefore treated as a required processed input for reproducing the final panel exactly.


In [ ]:
if not FINAL_ANNOTATED_PATH.exists():
    raise FileNotFoundError(
        "Final cleaned annotation file was not found:\n"
        f"{FINAL_ANNOTATED_PATH}\n"
        "Set FINAL_ANNOTATED_PATH to the manuscript annotation h5ad."
    )

adata = sc.read_h5ad(FINAL_ANNOTATED_PATH)
print(adata)
print(adata.obs["tech"].value_counts())
print(adata.obs["refined_combined_cluster"].value_counts())


## 7. Figure 2D | Integrated UMAP

The final UMAP is shown for all three technologies and colored either by technology or by the harmonized cell-type annotation.


In [ ]:
CELLTYPE_COLORS = {
    "Ex_L2/3_1": "#EF6702",
    "Ex_L2/3_2": "#f3a946",
    "Ex_L4/5": "#FFA900",
    "Ex_L5_1": "#5cb998",
    "Ex_L5_2": "#a0d4a5",
    "Ex_L6": "#00D893",
    "Ex_L7": "#93C9ED",
    "Ex_later": "#4CB3FF",

    "MSN_D1": "#807FFF",
    "MSN_D2": "#E618FF",
    "MSN_Otof": "#B34CFF",
    "MSN_Rreb1": "#742981",
    "Chat": "#C1A3CE",

    "In_Vip": "#F4B7C0",
    "In_Lamp5": "#F7EBA7",
    "In_Sst_1": "#70F1FC",
    "In_Sst_2": "#459BAD",
    "In_Pvalb_1": "#132EE4",
    "In_Pvalb_2": "#061584",

    "Oligo": "#EB6891",
    "OPC": "#A584F7",
    "Astro": "#006837",
    "Micro": "#FAD6EA",
    "Endo": "#DD8B59",
    "Peri": "#8A0179",
}

REGION_COLORS = {
    "Cortex": "#008000",
    "Striatum": "#8a2be2",
}


In [ ]:
# Recalculate UMAP from the existing neighbor graph using the manuscript setting.
sc.tl.umap(adata, min_dist=0.5)

sc.settings.set_figure_params(figsize=(8, 8))

sc.pl.umap(
    adata,
    color="tech",
    palette=TECH_COLORS,
    size=1,
    show=False,
)
plt.savefig(
    FIG_DIR / "Fig2D_integrated_UMAP_technology.pdf",
    bbox_inches="tight",
)
plt.close()

sc.pl.umap(
    adata,
    color="refined_combined_cluster",
    palette=CELLTYPE_COLORS,
    size=1,
    show=False,
)
plt.savefig(
    FIG_DIR / "Fig2D_integrated_UMAP_celltypes.pdf",
    bbox_inches="tight",
)
plt.close()


## 8. Figure 2D | Marker-gene dot plot

Marker expression is summarized across the harmonized integrated cell-type labels. The cluster dendrogram and annotation-color strip reproduce the visualization logic used in the source notebook.


In [ ]:
MARKERS = [
    "Slc17a7", "Lamp5", "Cux2", "Rorb", "Fezf2", "Slit2",
    "Tshz2", "Vwc2l", "Hs3st4", "Cdh18", "Ccn2", "Nxph4",
    "Slc17a6", "Gad1", "Pvalb", "Lhx6", "Sst", "Nos1", "Vip",
    "Drd1", "Drd2", "Adora2a", "Penk", "Otof", "Tac1",
    "Col11a1", "Rreb1", "Chat", "Slc5a7", "Cldn11", "Olig1",
    "Lhfpl3", "Gja1", "Csf1r", "Hexb", "Vtn", "Slco1a4",
    "Flt1", "Slc2a1",
]

GROUPBY = "refined_combined_cluster"

# Keep only markers available in the integrated matrix.
markers_available = [g for g in MARKERS if g in adata.var_names]
markers_missing = sorted(set(MARKERS) - set(markers_available))

print("Markers plotted:", len(markers_available))
if markers_missing:
    print("Markers absent from the integrated matrix:", markers_missing)

sc.tl.dendrogram(adata, groupby=GROUPBY)
ordered_clusters = adata.uns[f"dendrogram_{GROUPBY}"]["categories_ordered"]

sc.pl.dotplot(
    adata,
    var_names=markers_available,
    groupby=GROUPBY,
    dendrogram=True,
    cmap="Reds",
    swap_axes=True,
    mean_only_expressed=False,
    dot_max=0.6,
    vmax=3,
    figsize=(15, 16),
    show=False,
)

plt.subplots_adjust(wspace=0.01)
fig = plt.gcf()

# Scanpy places the main dot plot and dendrogram in separate axes.
ax_dend = fig.axes[1]

strip_colors = [
    CELLTYPE_COLORS.get(cluster, "#BDBDBD")
    for cluster in ordered_clusters
]
strip_rgb = np.array([to_rgb(c) for c in strip_colors])[np.newaxis, :, :]

strip_ax = fig.add_axes([
    ax_dend.get_position().x0,
    ax_dend.get_position().y1 + 0.01,
    ax_dend.get_position().width,
    0.03,
])
strip_ax.imshow(strip_rgb, aspect="auto")
strip_ax.set_xticks([])
strip_ax.set_yticks([])

plt.savefig(
    FIG_DIR / "Fig2D_marker_dotplot.pdf",
    bbox_inches="tight",
)
plt.show()


## 9. Technology composition of integrated clusters

This diagnostic reproduces the source workflow: cell counts are first normalized for technology-specific sampling depth and then normalized within each integrated cluster. It is useful for checking whether a harmonized cluster is dominated by one technology.


In [ ]:
def normalized_cluster_composition(
    adata,
    groupby="refined_combined_cluster",
    category="tech",
    category_order=None,
):
    ct = pd.crosstab(adata.obs[groupby], adata.obs[category])

    # Normalize each technology/category for sampling depth.
    p1 = ct.div(ct.sum(axis=0), axis=1)

    # Normalize within each integrated cluster.
    p2 = p1.div(p1.sum(axis=1), axis=0)

    sc.tl.dendrogram(adata, groupby=groupby)
    order = adata.uns[f"dendrogram_{groupby}"]["categories_ordered"]
    p2 = p2.reindex(order)

    if category_order is not None:
        p2 = p2[[x for x in category_order if x in p2.columns]]

    return p2


tech_composition = normalized_cluster_composition(
    adata,
    category="tech",
    category_order=["scrna", "h_bass", "t_bass"],
)

fig, ax = plt.subplots(figsize=(14, 4))

x = np.arange(len(tech_composition))
bottom = np.zeros(len(tech_composition))

for tech in tech_composition.columns:
    values = tech_composition[tech].to_numpy()
    ax.bar(
        x,
        values,
        bottom=bottom,
        color=TECH_COLORS[tech],
        width=1.0,
        edgecolor="none",
        label=tech,
    )
    bottom += values

ax.set_xlim(-0.5, len(tech_composition) - 0.5)
ax.set_ylim(0, 1)
ax.set_ylabel("Relative proportion")
ax.set_xlabel("Integrated cell type")
ax.set_xticks(x)
ax.set_xticklabels(tech_composition.index, rotation=45, ha="right")
ax.grid(False)
ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.savefig(
    FIG_DIR / "Fig2D_integrated_cluster_technology_composition.pdf",
    bbox_inches="tight",
)
plt.show()


## 10. Annotation agreement | technology-specific versus integrated labels

The original source notebook visualized row-normalized contingency tables between the pre-integration annotation and the Harmony-derived integrated annotation. The functions below retain that diagnostic and additionally calculate the true Jaccard index between cell sets:

**J(A,B) = |A ∩ B| / |A ∪ B|**

where `A` is the set of cells assigned to a technology-specific label and `B` is the set of cells assigned to an integrated label within the same technology.

Edit `INDEPENDENT_LABEL_COLS` if the final technology-specific annotation column used for the manuscript differs from the defaults below. The source notebook explicitly used `refined_clusters` for scRNA-seq and tested both `refined_cluster` and `refined_cluster_3` for lower-plex BASSFISH.


In [ ]:
INDEPENDENT_LABEL_COLS = {
    "scrna": "refined_clusters",
    "h_bass": "refined_cluster",
    "t_bass": "refined_cluster",
}

INTEGRATED_LABEL_COL = "refined_combined_cluster"


def annotation_fraction_table(obs, independent_col, integrated_col):
    """Row-normalized contingency table used in the source notebook."""
    table = pd.crosstab(obs[independent_col], obs[integrated_col])
    return table.div(table.sum(axis=1), axis=0)


def annotation_jaccard_table(obs, independent_col, integrated_col):
    """Jaccard similarity between cell sets defined by two annotations."""
    df = obs[[independent_col, integrated_col]].dropna().copy()
    df[independent_col] = df[independent_col].astype(str)
    df[integrated_col] = df[integrated_col].astype(str)

    intersection = pd.crosstab(df[independent_col], df[integrated_col])
    row_size = intersection.sum(axis=1)
    col_size = intersection.sum(axis=0)

    result = intersection.astype(float).copy()

    for row in intersection.index:
        for col in intersection.columns:
            inter = float(intersection.loc[row, col])
            union = float(row_size.loc[row] + col_size.loc[col] - inter)
            result.loc[row, col] = inter / union if union > 0 else np.nan

    return result


def plot_annotation_heatmap(
    table,
    title,
    out_file,
    cmap="Reds",
    vmin=0,
    vmax=1,
):
    fig_w = max(6, 0.28 * table.shape[1] + 3)
    fig_h = max(6, 0.22 * table.shape[0] + 3)

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))

    sns.heatmap(
        table,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        annot=False,
        linewidths=0,
        cbar=True,
        ax=ax,
    )

    for spine in ax.spines.values():
        spine.set_visible(False)

    if ax.collections:
        ax.collections[0].set_edgecolor("face")

    ax.tick_params(axis="both", which="both", length=0)
    ax.set_title(title)
    ax.set_xlabel("Integrated Harmony annotation")
    ax.set_ylabel("Technology-specific annotation")

    plt.tight_layout()
    plt.savefig(out_file, bbox_inches="tight")
    plt.show()


In [ ]:
annotation_tables = {}

for tech, independent_col in INDEPENDENT_LABEL_COLS.items():
    obs_tech = adata.obs.loc[adata.obs["tech"] == tech].copy()

    if independent_col not in obs_tech.columns:
        print(
            f"[SKIP] {tech}: independent annotation column "
            f"'{independent_col}' is not available."
        )
        continue

    if INTEGRATED_LABEL_COL not in obs_tech.columns:
        print(
            f"[SKIP] {tech}: integrated annotation column "
            f"'{INTEGRATED_LABEL_COL}' is not available."
        )
        continue

    fraction = annotation_fraction_table(
        obs_tech,
        independent_col,
        INTEGRATED_LABEL_COL,
    )

    jaccard = annotation_jaccard_table(
        obs_tech,
        independent_col,
        INTEGRATED_LABEL_COL,
    )

    annotation_tables[tech] = {
        "fraction": fraction,
        "jaccard": jaccard,
    }

    fraction.to_csv(
        OUT_DIR / f"{tech}_annotation_row_fraction.csv"
    )
    jaccard.to_csv(
        OUT_DIR / f"{tech}_annotation_jaccard.csv"
    )

    plot_annotation_heatmap(
        jaccard,
        title=f"{tech}: independent vs integrated annotation (Jaccard)",
        out_file=FIG_DIR / f"Fig2D_{tech}_annotation_Jaccard.pdf",
        vmin=0,
        vmax=1,
    )
